In [ ]:
import os
import shutil
import pandas as pd
import gc
import json
from IPython.display import clear_output
from dotenv import load_dotenv
from groq import Groq
from agents.manager_agent import ManagerAgent
from llm_loader import LLMManager
import config

# ==========================================
# 🔐 BẢO MẬT: NẠP API KEY TỪ FILE .ENV
# ==========================================
load_dotenv() # Tự động tìm và load các biến từ file .env
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("❌ LỖI: Chưa tìm thấy GROQ_API_KEY trong file .env. Hãy thêm nó vào trước khi chạy!")

# Khởi tạo Groq Client
groq_client = Groq(api_key=groq_api_key)

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN & GIỚI HẠN
# ==========================================
EXCEL_FILE_PATH = "Thư mục KQ/VB gốc/variant B.xlsx"
OUTPUT_EXCEL_PATH = "Thư mục KQ/VB sau tóm tắt/test_results.xlsx"
BATCH_BACKUP_DIR = "Thư mục KQ/Data_Results"

MAX_DOCS = 20  # <--- Giới hạn cắt 200 văn bản mỗi lần chạy

os.makedirs(BATCH_BACKUP_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_EXCEL_PATH), exist_ok=True)

df = pd.read_excel(EXCEL_FILE_PATH)
df = df.head(MAX_DOCS)

if 'Summary' not in df.columns:
    df['Summary'] = ""
if 'Result' not in df.columns:
    df['Result'] = ""

llm = LLMManager()

print(f"✅ Đã tải file Excel. Tổng số văn bản sẽ xử lý: {len(df)}")
print("🚀 Bắt đầu quá trình treo máy tự động...")

# ==========================================
# HÀM CHẤM ĐIỂM BẰNG GROQ API (PROMPT CHUYÊN SÂU)
# ==========================================
def evaluate_summary_with_groq(article, summary):
    prompt = f"""Bạn là một Chuyên gia Ngôn ngữ học và Đánh giá AI khắt khe, chuyên kiểm định chất lượng tóm tắt văn bản tiếng Việt.
Nhiệm vụ của bạn là đánh giá bản tóm tắt bên dưới dựa trên văn bản gốc. Hãy chấm điểm từ 0.0 (Tệ nhất) đến 1.0 (Hoàn hảo) cho 3 tiêu chí sau:

1. Faithfulness (Tính trung thực): Bản tóm tắt có bịa đặt (hallucinate) thêm thông tin, sai số liệu, hay xuyên tạc ý nghĩa của văn bản gốc không?
2. Relevance (Tính bao quát): Bản tóm tắt có bao hàm đủ các ý chính quan trọng nhất không? Có bỏ sót thông điệp cốt lõi nào không?
3. Coherence & Conciseness (Tính mạch lạc và súc tích): Bản tóm tắt có trôi chảy, logic, dễ đọc và loại bỏ được các chi tiết rườm rà/lặp từ không?

[VĂN BẢN GỐC]
{article}

[BẢN TÓM TẮT CỦA AI]
{summary}

⚠️ YÊU CẦU BẮT BUỘC VỀ ĐẦU RA:
- Bạn phải trả về kết quả định dạng JSON. KHÔNG xuất thêm bất kỳ giải thích, markdown (như ```json) hay lời chào nào bên ngoài JSON.
- Các trường "reason" phải được viết bằng tiếng Việt, phân tích sắc bén, chỉ đích danh lỗi sai (nếu có) để giải thích tại sao lại cho mức điểm đó.
- Cấu trúc JSON phải chính xác tuyệt đối như sau:
{{
  "faithfulness_score": <float>,
  "faithfulness_reason": "<string>",
  "relevance_score": <float>,
  "relevance_reason": "<string>",
  "coherence_conciseness_score": <float>,
  "coherence_conciseness_reason": "<string>"
}}"""

    try:
        response = groq_client.chat.completions.create(
            messages=[
                {"role": "system", "content": "You are a JSON-generating evaluation AI. Output ONLY valid JSON matching the user's schema."},
                {"role": "user", "content": prompt}
            ],
            model="llama-3.3-70b-versatile", # Dùng model 70B cực kỳ thông minh của Groq để làm trọng tài
            temperature=0.1,         # Nhiệt độ thấp để đánh giá khách quan, ổn định
            response_format={"type": "json_object"} # Cờ ép trả về JSON chuẩn
        )
        
        # Đọc chuỗi JSON trả về
        result_str = response.choices[0].message.content
        return json.loads(result_str)
    except Exception as e:
        print(f"Lỗi khi gọi Groq API: {e}")
        return None

# ==========================================
# VÒNG LẶP TỰ ĐỘNG XỬ LÝ TỪNG VĂN BẢN
# ==========================================
for index, row in df.iterrows():
    text_id = index + 1
    input_text = str(row.iloc[0]) 
    
    if pd.isna(input_text) or len(input_text.strip()) < 10:
        continue

    clear_output(wait=True)
    print(f"⏳ ĐANG XỬ LÝ VĂN BẢN SỐ: {text_id} / {len(df)} (Dòng {index + 2} trong Excel)")
    print(f"Tiến trình tổng: {round((text_id/len(df))*100, 1)}%")
    print("="*60)

    # 1. Reset môi trường data
    if os.path.exists(config.DATA_DIR):
        shutil.rmtree(config.DATA_DIR)
    os.makedirs(config.DATA_DIR)

    # 2. Chạy workflow sinh tóm tắt
    manager = ManagerAgent()
    try:
        final_output = manager.run_workflow(input_text=input_text, style="news_brief", resume=False)
        summary_result = final_output.get("summary", "")
    except Exception as e:
        summary_result = f"LỖI: {str(e)}"
        print(f"❌ Phát hiện lỗi tại văn bản {text_id}: {str(e)}")

    # 3. CHẤM ĐIỂM BẰNG GROQ API
    print("🤖 Đang gọi Groq API (Llama3-70B) để thẩm định chất lượng tóm tắt...")
    eval_result = evaluate_summary_with_groq(input_text, summary_result)
    
    # 4. ĐÓNG GÓI JSON KẾT QUẢ ĐẦU RA THEO ĐÚNG YÊU CẦU
    final_json_dict = {
        "id": text_id,
        "style": "news_brief",
        "article": input_text,
        "summary": summary_result,
        "faithfulness_score": "N/A",
        "faithfulness_reason": "Lỗi API",
        "relevance_score": "N/A",
        "relevance_reason": "Lỗi API",
        "coherence_conciseness_score": "N/A",
        "coherence_conciseness_reason": "Lỗi API"
    }

    # Nếu Groq trả về kết quả thành công, cập nhật đè lên các giá trị N/A
    if eval_result:
        final_json_dict.update({
            "faithfulness_score": eval_result.get("faithfulness_score", "N/A"),
            "faithfulness_reason": eval_result.get("faithfulness_reason", ""),
            "relevance_score": eval_result.get("relevance_score", "N/A"),
            "relevance_reason": eval_result.get("relevance_reason", ""),
            "coherence_conciseness_score": eval_result.get("coherence_conciseness_score", "N/A"),
            "coherence_conciseness_reason": eval_result.get("coherence_conciseness_reason", "")
        })

    # Chuyển Dict thành chuỗi định dạng JSON đẹp mắt, hỗ trợ tiếng Việt
    final_result_json = json.dumps(final_json_dict, ensure_ascii=False, indent=2)

    # 5. Ghi Excel và Lưu liên tục
    df.at[index, 'Summary'] = summary_result
    df.at[index, 'Result'] = final_result_json
    df.to_excel(OUTPUT_EXCEL_PATH, index=False)
    print(f"✅ Đã ghi thành công kết quả và cục JSON Đánh giá vào file Excel.")

    # 6. Sao lưu Session & Log
    backup_folder = os.path.join(BATCH_BACKUP_DIR, f"text_{text_id}")
    if os.path.exists(backup_folder):
        shutil.rmtree(backup_folder)
    shutil.copytree(config.DATA_DIR, backup_folder)

    # 7. DỌN DẸP BỘ NHỚ
    llm.unload()
    del manager      
    gc.collect()     

print("\n🎉 HOÀN TẤT! QUÁ TRÌNH TÓM TẮT & CHẤM ĐIỂM HÀNG LOẠT ĐÃ XONG!")